# 🌸 Flower Federated Learning Training Pipeline (FL-IoT-IDS)
## Industrial Edge Intrusion Detection with Flower (`flwr`)
### Master's Thesis: *Xây dựng và đánh giá prototype hệ thống phát hiện xâm nhập IoT sử dụng Federated Learning trong môi trường dữ liệu non-IID*
**Author:** Nguyễn Việt Kỳ Quân (2026) | **Institution:** HCMUT - VNU-HCM

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVKQ2022/FedLearning/blob/main/notebooks/flower_federated_pipeline.ipynb)

---

### 🎯 Purpose in Thesis
This notebook implements the industrial **Flower framework (`flwr`)** orchestration layer for our thesis prototype. It demonstrates how decentralized IoT edge intrusion detection can be deployed using industry-standard federated learning protocols:

1. **`FlowerIoTClient`:** Edge client adhering to `flwr.client.NumPyClient`, implementing local training with FedProx proximal loss $\frac{\mu}{2}\|w - w_t\|^2$ and client parameter drift norm tracking.
2. **`FlowerIoTServerStrategy`:** Custom server strategy extending Flower's `FedAvg` and `FedProx` with centralized holdout evaluation, communication payload tracking ($2 \times \sum m_t \times B$), and model checkpointing.
3. **Heterogeneous Evaluation:** Rigorously benchmarks IID vs Non-IID Dirichlet ($\alpha = 0.5, 0.1$) distributions on the CICIoT2023 dataset.

---
### 🛠️ Step 1: Environment Setup & Flower Verification
We verify GPU acceleration, install `flwr`, synchronize code in Colab, and enforce deterministic seeds.

In [ ]:
import os, sys, subprocess, warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Compute Device: {device} | PyTorch: {torch.__version__}')

# Colab setup
if os.path.exists('/content'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass
    repo_dir = '/content/FedLearning'
    if not os.path.exists(repo_dir):
        subprocess.run(['git', 'clone', 'https://github.com/NVKQ2022/FedLearning.git', repo_dir], check=True)
    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
elif os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

try:
    import flwr as fl
    print(f'✅ Flower Version: {fl.__version__}')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts', 'flwr>=1.13.0'], check=True)
    import flwr as fl
    print(f'✅ Flower installed: {fl.__version__}')

from src.utils.seed import set_seed
set_seed(42)
print('✅ Deterministic random seed locked at 42.')

---
### ⚙️ Step 2: Global Hyperparameter Configuration
Configure scenario settings: number of edge clients, communication rounds, local epochs, and FedProx coefficient $\mu$.

In [ ]:
from src.utils.config import ExperimentConfig

CONFIG = ExperimentConfig(
    experiment_name="flower_e5_fedprox_noniid",
    seed=42,
    
    # Dataset & Preprocessing
    sample_size=100000,                 # 100k samples for fast execution
    test_size=0.2,                      # 20% holdout test set
    val_size=0.1,                       # 10% centralized server validation set
    scaler_type="robust",               # 'robust' (median/IQR)
    batch_size=64,                      # Client local batch size
    
    # Architecture & Regularization
    hidden_dims=(128, 64),              # Lightweight edge MLP (~15k params)
    dropout_rate=0.2,
    
    # Loss Function
    loss_type="focal_loss",             # MultiClassFocalLoss
    class_weight_strategy="balanced",   # Inverse frequency class weighting
    focal_gamma=2.0,
    
    # Optimization
    optimizer_type="adamw",
    learning_rate=1e-3,
    weight_decay=1e-4,
    max_grad_norm=5.0,
    
    # Flower Federated Parameters
    num_clients=5,                      # Number of virtual IoT edge clients K
    num_rounds=10,                      # Server communication rounds T
    local_epochs=2,                     # Client local epochs per round E
    federated_strategy="fedprox",       # 'fedprox' or 'fedavg'
    mu=0.05,                            # FedProx proximal coefficient (mu=0.0 for FedAvg)
    partition_type="dirichlet",         # 'dirichlet' or 'iid'
    dirichlet_alpha=0.1                 # Dirichlet concentration (0.1 = severe skew)
)

CONFIG.display()

---
### 💾 Step 3: Leak-Free Preprocessing & Partitioning
We preprocess the dataset and generate client partitions.

In [ ]:
from src.data.preprocess import load_and_preprocess_ciciot2023, compute_balanced_class_weights
from src.data.partitioner import (
    StratifiedIIDPartitioner,
    DirichletNonIIDPartitioner,
    create_client_dataloaders,
    summarize_client_partitions
)

# Locate dataset CSV
candidate_paths = [
    CONFIG.gdrive_csv,
    CONFIG.alt_gdrive_csv,
    CONFIG.target_csv_path,
    'datasets/CICIOT2023/merged_CICIOT2023_data.csv',
    'datasets/merged_CICIOT2023_data.csv',
]
csv_path = next((p for p in candidate_paths if os.path.exists(p) and os.path.getsize(p) > 10*1024*1024), None)
if not csv_path:
    csv_path = CONFIG.target_csv_path

print(f"📁 Loading and preprocessing from: {csv_path}")
data = load_and_preprocess_ciciot2023(
    csv_path=csv_path,
    test_size=CONFIG.test_size,
    val_size=CONFIG.val_size,
    sample_size=CONFIG.sample_size,
    scaler_type=CONFIG.scaler_type,
    batch_size=CONFIG.batch_size,
    random_state=CONFIG.seed
)

class_weights = compute_balanced_class_weights(data['y_train'], data['num_classes'], strategy=CONFIG.class_weight_strategy)
class_names = data['class_names']
minority_classes = ['Web-based', 'Brute-force']

# Generate client partitions
if CONFIG.partition_type == "iid":
    partitioner = StratifiedIIDPartitioner(num_clients=CONFIG.num_clients, seed=CONFIG.seed)
else:
    partitioner = DirichletNonIIDPartitioner(
        num_clients=CONFIG.num_clients,
        alpha=CONFIG.dirichlet_alpha,
        min_samples_per_client=100,
        seed=CONFIG.seed
    )

client_partitions = partitioner.partition(data['X_train'], data['y_train'])
client_loaders = create_client_dataloaders(data['X_train'], data['y_train'], client_partitions, batch_size=CONFIG.batch_size)

summary_df = summarize_client_partitions(client_partitions, data['y_train'], class_names=class_names)
display(summary_df[['client_id', 'total_samples', 'dominant_class', 'dominant_pct', 'entropy']])

---
### 🏛️ Step 4: Model Backbone & Server Evaluation Callback
We initialize the global `TabularIoTMLPModel`, loss function, and the centralized server holdout evaluation callback (`build_flower_server_eval_fn`).

In [ ]:
from src.models.tabular_mlp import TabularIoTMLPModel
from src.losses.focal_loss import build_loss_function
from src.federated.flower_server import build_flower_server_eval_fn

global_model = TabularIoTMLPModel(
    input_dim=data['input_dim'],
    hidden_dims=CONFIG.hidden_dims,
    num_classes=data['num_classes'],
    dropout_rate=CONFIG.dropout_rate
).to(device)

criterion = build_loss_function(
    loss_type=CONFIG.loss_type,
    class_weights=class_weights if CONFIG.class_weight_strategy != 'none' else None,
    gamma=CONFIG.focal_gamma,
    device=device
)

model_size_bytes = sum(p.numel() * p.element_size() for p in global_model.parameters())

# Server-side holdout evaluation callback executed after every communication round
flower_eval_fn = build_flower_server_eval_fn(
    model=global_model,
    val_loader=data['val_loader'],
    class_names=class_names,
    criterion=criterion,
    minority_classes=minority_classes,
    device=device
)

print(f"✅ Global Model: {global_model.get_num_parameters():,} params ({model_size_bytes/1024:.2f} KB)")
print("✅ Server Holdout Evaluation Callback built successfully.")

---
### 🌸 Step 5: Flower Strategy & ClientApp Configuration
We initialize `FlowerIoTServerStrategy` and define `FlowerIoTClient` wrapped inside Flower's `ClientApp`.

In [ ]:
import flwr as fl
from src.federated.flower_server import FlowerIoTServerStrategy
from src.federated.flower_client import FlowerIoTClient

os.makedirs('checkpoints', exist_ok=True)

# 1. Initialize Flower Strategy
strategy = FlowerIoTServerStrategy(
    strategy_name=CONFIG.federated_strategy,
    mu=CONFIG.mu if CONFIG.federated_strategy == 'fedprox' else 0.0,
    fraction_fit=1.0,
    min_fit_clients=CONFIG.num_clients,
    min_available_clients=CONFIG.num_clients,
    local_epochs=CONFIG.local_epochs,
    model_size_bytes=float(model_size_bytes),
    evaluate_fn=flower_eval_fn,
    checkpoint_dir="checkpoints"
)

# 2. Define ClientApp factory
def client_fn(context_or_cid) -> fl.client.Client:
    if hasattr(context_or_cid, "node_config"):
        client_id = int(context_or_cid.node_config.get("partition-id", getattr(context_or_cid, "node_id", 0)))
    else:
        client_id = int(context_or_cid)
        
    local_model = TabularIoTMLPModel(
        input_dim=data['input_dim'],
        hidden_dims=CONFIG.hidden_dims,
        num_classes=data['num_classes'],
        dropout_rate=CONFIG.dropout_rate
    )
    client = FlowerIoTClient(
        client_id=client_id,
        model=local_model,
        train_loader=client_loaders[client_id],
        val_loader=data['val_loader'],
        device="cpu"  # CPU execution prevents GPU memory fragmentation in simulation
    )
    return client.to_client()

try:
    from flwr.client import ClientApp
    client_app = ClientApp(client_fn=client_fn)
except ImportError:
    client_app = None

print(f"✅ Flower Server Strategy: {strategy.strategy_name.upper()} (mu={strategy.mu})")
print("✅ Flower Client factory ready.")

---
### 🚀 Step 6: Flower Federated Simulation Execution
We launch the multi-round federated training simulation using Flower's simulation engine.

In [ ]:
import time

# Memory safe environment settings
os.environ["RAY_memory_monitor_refresh_ms"] = "0"
os.environ["RAY_memory_usage_threshold"] = "0.99"

print(f"Starting Flower Federated Simulation: {CONFIG.num_clients} clients, {CONFIG.num_rounds} rounds...")
print(f"Strategy: {strategy.strategy_name.upper()} | mu: {strategy.mu} | Local Epochs: {strategy.local_epochs}")
print("=" * 80)

sim_start = time.perf_counter()

# Run simulation
client_resources = {"num_cpus": 1, "num_gpus": 0.0}

try:
    from flwr.server import ServerApp, ServerConfig
    from flwr.simulation import run_simulation
    server_app = ServerApp(server_fn=lambda ctx: fl.server.ServerAppComponents(strategy=strategy, config=ServerConfig(num_rounds=CONFIG.num_rounds)))
    run_simulation(
        server_app=server_app,
        client_app=client_app,
        num_supernodes=CONFIG.num_clients,
        backend_config={"client_resources": client_resources}
    )
except Exception as e:
    print(f"Falling back to classic start_simulation: {e}")
    fl.simulation.start_simulation(
        client_fn=client_fn,
        num_clients=CONFIG.num_clients,
        config=fl.server.ServerConfig(num_rounds=CONFIG.num_rounds),
        strategy=strategy,
        client_resources=client_resources
    )

sim_time = time.perf_counter() - sim_start
print("=" * 80)
print(f"🎉 Flower Simulation finished in {sim_time:.1f}s ({sim_time/60:.2f} min)!")
print(f"Best Round: {strategy.best_round} | Best Val Macro-F1: {strategy.best_macro_f1 * 100:.2f}%")

---
### 📈 Step 7: Convergence & Communication Overhead Tracking
We inspect the Validation Macro-F1 progression, cumulative communication payload, and client parameter drift.

In [ ]:
import matplotlib.pyplot as plt

rh = strategy.round_history
rounds = rh.get("round", [])

if len(rounds) > 0:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=120)
    
    # Left: Macro-F1 Convergence
    val_f1 = rh["val_macro_f1"][-len(rounds):]
    ax1.plot(rounds, [f * 100 for f in val_f1], 'o-', color='#2980b9', linewidth=2)
    ax1.set_title(f"Flower {strategy.strategy_name.upper()} Macro-F1 Convergence", fontsize=11, fontweight='bold')
    ax1.set_xlabel("Round", fontsize=10)
    ax1.set_ylabel("Macro-F1 (%)", fontsize=10)
    ax1.grid(True, linestyle="--", alpha=0.5)
    
    # Right: Cumulative Communication Overhead (MB)
    ax2.plot(rounds, rh["comm_cost_mb"], 's-', color='#27ae60', linewidth=2)
    ax2.set_title("Cumulative Network Communication Overhead (MB)", fontsize=11, fontweight='bold')
    ax2.set_xlabel("Round", fontsize=10)
    ax2.set_ylabel("Transferred (MB)", fontsize=10)
    ax2.grid(True, linestyle="--", alpha=0.5)
    
    plt.tight_layout()
    plt.show()
    print(f"📊 Total Payload Exchanged: {rh['comm_cost_mb'][-1]:.2f} MB")

---
### 🏆 Step 8: Holdout Test Evaluation
We evaluate the best model weights on the stationary holdout test set.

In [ ]:
from src.evaluation.evaluator import evaluate_comprehensive
from src.visualization import plot_confusion_matrix

# Restore best global weights from checkpoint
best_ckpt = f"checkpoints/flower_{strategy.strategy_name}_best_weights.npz"
if os.path.exists(best_ckpt):
    npz = np.load(best_ckpt)
    best_weights = [npz[f"arr_{i}"] for i in range(len(npz.files))]
    global_model.set_weights(best_weights)
    print(f"✅ Restored best weights from: {best_ckpt}")

test_results = evaluate_comprehensive(
    model=global_model,
    dataloader=data['test_loader'],
    class_names=class_names,
    minority_classes=minority_classes,
    device=device
)

print('=' * 65)
print(f"🏆 FLOWER {strategy.strategy_name.upper()} FINAL HOLDOUT TEST EVALUATION")
print('=' * 65)
print(f"Test Accuracy:    {test_results['accuracy'] * 100:.2f}%")
print(f"Macro-F1 Score:   {test_results['macro_f1'] * 100:.2f}%")
print(f"Weighted-F1:      {test_results['weighted_f1'] * 100:.2f}%")
print(f"Macro-Precision:  {test_results['macro_precision'] * 100:.2f}%")
print(f"Macro-Recall:     {test_results['macro_recall'] * 100:.2f}%")
print('-' * 65)
print('Minority Class Recall (Stealth Infiltration Intrusions):')
for cls, rec in test_results['minority_recall'].items():
    print(f"   • {cls:<15}: {rec * 100:.2f}%")
print('=' * 65)

# Render Confusion Matrix
plot_confusion_matrix(
    cm=test_results['confusion_matrix'],
    class_names=class_names,
    title=f"Flower {strategy.strategy_name.upper()}: Confusion Matrix",
    show=True
)

---
### 💾 Step 9: Export Scenario Results
We persist scenario partitions and server evaluation metrics to `scenarios/<scenario_name>/`.

In [ ]:
from src.federated.scenario import setup_scenario_hierarchy
import json

scenario_dir = f"scenarios/{CONFIG.experiment_name}"
setup_scenario_hierarchy(
    scenario_name=CONFIG.experiment_name,
    client_partitions=client_partitions,
    X=data['X_train'],
    y=data['y_train'],
    class_names=class_names,
    scenarios_root="scenarios"
)

metrics_export = {k: v for k, v in test_results.items() if k != 'confusion_matrix'}
metrics_export['round_history'] = strategy.round_history
metrics_path = f"{scenario_dir}/server/metrics.json"

with open(metrics_path, 'w') as f:
    json.dump(metrics_export, f, indent=2)

print(f"✅ Successfully exported scenario artifacts to: {scenario_dir}/")
print(f"   - Server Metrics: {metrics_path}")
print(f"\n🎉 Flower {strategy.strategy_name.upper()} Pipeline execution complete!")